# Experiment 11: Deep Autoencoder Anomaly Detection

## 1. Overview & Architecture
Deep Neural Autoencoders learn an efficient low-dimensional manifold representation of **normal benign UAV operations**.
- During inference on test data, the reconstruction error $E(x) = \frac{1}{D} \sum_{j=1}^D (x_j - \hat{x}_j)^2$ measures how far the sample deviates from normal behavior.
- High reconstruction error directly flags cyber and physical attacks.
- **Architecture**: Multi-Layer Perceptron Bottleneck Autoencoder ($D \to 32 \to 16 \to 8 \to 16 \to 32 \to D$).


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

from utils.data_loader import load_physical_dataset, load_cyber_dataset, get_novelty_detection_split
from utils.unsupervised_metrics import evaluate_anomaly_detector, measure_inference_speed


## 2. Train Physical and Cyber Deep Autoencoders


In [ ]:
def train_autoencoder(X_train, hidden_dims=[32, 16, 8], epochs=25):
    input_dim = X_train.shape[1]
    enc_layers = [layers.Input(shape=(input_dim,))]
    for h in hidden_dims:
        enc_layers.append(layers.Dense(h, activation='relu'))
    for h in reversed(hidden_dims[:-1]):
        enc_layers.append(layers.Dense(h, activation='relu'))
    enc_layers.append(layers.Dense(input_dim, activation='linear'))
    
    model = keras.Sequential(enc_layers)
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train, X_train, epochs=epochs, batch_size=64, verbose=0)
    return model

# 1. Physical Autoencoder
X_p, y_p, _ = load_physical_dataset("../Physical_UAV_Dataset.csv")
X_p_tr, X_p_te, y_p_te_bin, y_p_te_multi = get_novelty_detection_split(X_p, y_p)
scaler_p = StandardScaler()
X_p_tr_s = scaler_p.fit_transform(X_p_tr)
X_p_te_s = scaler_p.transform(X_p_te)

ae_phys = train_autoencoder(X_p_tr_s, hidden_dims=[16, 8, 4])
tr_p_err = np.mean((X_p_tr_s - ae_phys.predict(X_p_tr_s, verbose=0))**2, axis=1)
te_p_err = np.mean((X_p_te_s - ae_phys.predict(X_p_te_s, verbose=0))**2, axis=1)
m_p_opt, _, _ = evaluate_anomaly_detector(te_p_err, y_p_te_bin, y_p_te_multi, "Autoencoder (Best F1)", "Physical")
tau_p_5 = np.percentile(tr_p_err, 95)
m_p_5, _, _ = evaluate_anomaly_detector(te_p_err, y_p_te_bin, y_p_te_multi, "Autoencoder (5% FAR)", "Physical", threshold=tau_p_5)

# 2. Cyber Autoencoder
X_c, y_c, _ = load_cyber_dataset("../Cyber_UAV_Dataset.csv")
X_c_tr, X_c_te, y_c_te_bin, y_c_te_multi = get_novelty_detection_split(X_c, y_c)
scaler_c = StandardScaler()
X_c_tr_s = scaler_c.fit_transform(X_c_tr)
X_c_te_s = scaler_c.transform(X_c_te)

ae_cyb = train_autoencoder(X_c_tr_s, hidden_dims=[32, 16, 8])
tr_c_err = np.mean((X_c_tr_s - ae_cyb.predict(X_c_tr_s, verbose=0))**2, axis=1)
te_c_err = np.mean((X_c_te_s - ae_cyb.predict(X_c_te_s, verbose=0))**2, axis=1)
m_c_opt, _, _ = evaluate_anomaly_detector(te_c_err, y_c_te_bin, y_c_te_multi, "Autoencoder (Best F1)", "Cyber")
tau_c_5 = np.percentile(tr_c_err, 95)
m_c_5, _, _ = evaluate_anomaly_detector(te_c_err, y_c_te_bin, y_c_te_multi, "Autoencoder (5% FAR)", "Cyber", threshold=tau_c_5)

pd.DataFrame([m_p_opt, m_p_5, m_c_opt, m_c_5])


## 3. Reconstruction Error Distributions (Log Scale)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_p = pd.DataFrame({'Log_MSE': np.log10(te_p_err + 1e-4), 'Class': y_p_te_multi})
sns.boxplot(data=df_p, x='Class', y='Log_MSE', palette='Set2', ax=axes[0])
axes[0].set_title("Physical Autoencoder: Log Reconstruction Error", fontsize=13, fontweight='bold')
axes[0].set_ylabel("log10(Reconstruction Error)")
axes[0].grid(True, alpha=0.3)

df_c = pd.DataFrame({'Log_MSE': np.log10(te_c_err + 1e-4), 'Class': y_c_te_multi})
sns.boxplot(data=df_c, x='Class', y='Log_MSE', palette='Set2', ax=axes[1])
axes[1].set_title("Cyber Autoencoder: Log Reconstruction Error", fontsize=13, fontweight='bold')
axes[1].set_ylabel("log10(Reconstruction Error)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Key Findings
1. **Cyber Autoencoder Success**: Deep Autoencoder on Cyber network data achieves **92.92% ROC-AUC** and **83.67% DoS recall** at a strict 5% False Alarm Rate.
2. **Physical Autoencoder Robustness**: 100% detection of Evil_Twin and FDI with only 4.2% False Alarm Rate on Benign flight data.
